In [20]:
import os
import pandas as pd

# Get all files in the current directory
files = os.listdir("logs")

# Filter files that don't contain "mouse"
non_mouse_files = [file.replace("_log.bowtie2", "") for file in files 
                  if "mouse" not in file.lower() and "_log.bowtie2" in file]

non_mouse_files.sort()
non_mouse_files[1:3]


['clone32__HU__G9a__rep2', 'clone32__HU__G9a__rep3']

In [21]:
def extract_mapped_reads(bowtie_file_path):
    """
    Extract the sum of mapped reads from a bowtie2 log file.
    
    Args:
        bowtie_file_path (str): Path to the bowtie2 log file
        
    Returns:
        int: Sum of mapped reads (concordantly aligned exactly 1 time + >1 times)
        float: Overall alignment rate percentage
    """
    try:
        with open(bowtie_file_path, 'r') as file:
            content = file.read()
            
            # Extract reads aligned exactly 1 time
            aligned_once_line = [line for line in content.split('\n') if "aligned concordantly exactly 1 time" in line][0]
            aligned_once = int(aligned_once_line.split()[0])
            
            # Extract reads aligned more than 1 time
            aligned_multiple_line = [line for line in content.split('\n') if "aligned concordantly >1 times" in line][0]
            aligned_multiple = int(aligned_multiple_line.split()[0])
            
            # Calculate sum of mapped reads
            sum_mapped_reads = aligned_once + aligned_multiple
            
            # Extract overall alignment rate
            alignment_rate_line = [line for line in content.split('\n') if "overall alignment rate" in line][0]
            alignment_rate = float(alignment_rate_line.split('%')[0])
            
            return sum_mapped_reads, alignment_rate
            
    except Exception as e:
        print(f"Error processing file {bowtie_file_path}: {str(e)}")
        return 0, 0.0

In [24]:
# Create a list to store the results
results = []

# Process each file
for file in non_mouse_files:
    try:
        human_reads, human_rate = extract_mapped_reads("logs/" + file + "_log.bowtie2")
        mouse_reads, mouse_rate = extract_mapped_reads("logs/" + file + "_mouse_log.bowtie2")
        
        results.append({
            'file': file,
            'human_reads': human_reads,
            'human_alignment_pct': human_rate,
            'mouse_reads': mouse_reads,
            'mouse_alignment_pct': mouse_rate,
            'human_mouse_ratio': human_reads/mouse_reads/100 if mouse_reads > 0 else float('inf')
        })
    except Exception as e:
        print(f"Error processing {file}: {str(e)}")

# Create a pandas DataFrame
df = pd.DataFrame(results)

# Format the table
pd.set_option('display.float_format', '{:.2f}'.format)
print(df)

                                     file  human_reads  human_alignment_pct  \
0                  clone32__HU__G9a__rep1      6958006                95.87   
1                  clone32__HU__G9a__rep2      6585925                95.73   
2                  clone32__HU__G9a__rep3      5399598                96.25   
3              clone32__HU__H3K27ac__rep1      7115544                93.30   
4              clone32__HU__H3K27ac__rep2      8319911                93.81   
..                                    ...          ...                  ...   
70            cloneA3__Unt__H3K9me3__rep2      2975616                93.91   
71                cloneA3__Unt__IgG__rep1      6908177                96.19   
72  cloneA3__Unt_pA_Mnase__H3K27me3__rep1     51697493                99.66   
73   cloneA3__Unt_pA_Mnase__H3K9me2__rep1      4647500                97.92   
74       cloneA3__Unt_pA_Mnase__IgG__rep1      3628934                98.81   

    mouse_reads  mouse_alignment_pct  human_mouse_r

In [25]:
df.to_csv('mapping_results.csv', index=False)